# 第7章：建立聊天應用程式
## OpenAI API 快速入門

本筆記本改編自[Azure OpenAI 範例庫](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst)，其中包含存取[Azure OpenAI](notebook-azure-openai.ipynb)服務的筆記本。

Python OpenAI API 同樣可搭配 Azure OpenAI 模型使用，只需做少許修改。詳細了解差異請參考：[如何使用 Python 切換 OpenAI 和 Azure OpenAI 端點](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)


# 概覽  
「大型語言模型是一種將文本映射到文本的函數。給定一段輸入文字，大型語言模型嘗試預測接下來會出現的文字」(1)。這個「快速入門」筆記本將向使用者介紹大型語言模型的高層次概念、開始使用 AML 的核心套件需求、對提示設計的簡單介紹，以及多個不同使用案例的短範例。 


## 目錄  

[概覽](#overview)  
[如何使用 OpenAI 服務](#how-to-use-openai-service)  
[1. 建立您的 OpenAI 服務](#1.-creating-your-openai-service)  
[2. 安裝](#2.-installation)    
[3. 憑證](#3.-credentials)  

[使用案例](#use-cases)    
[1. 摘要文本](#1.-summarize-text)  
[2. 分類文本](#2.-classify-text)  
[3. 生成新產品名稱](#3.-generate-new-product-names)  
[4. 微調分類器](#4.fine-tune-a-classifier)  

[參考資料](#references)


### 建立你的第一個提示  
這個簡短的練習將提供一個基本介紹，說明如何向 OpenAI 模型提交提示以完成一個簡單的任務「摘要」。  


<strong>步驟</strong>：  
1. 在你的 python 環境中安裝 OpenAI 庫  
2. 載入標準輔助庫並設定你為所建立的 OpenAI 服務所配置的典型 OpenAI 安全憑證  
3. 選擇一個適合你任務的模型  
4. 為模型建立一個簡單的提示  
5. 將你的請求提交到模型 API！  


### 1. 安裝 OpenAI


In [ ]:
%pip install openai python-dotenv

### 2. 匯入輔助程式庫及實例化憑證


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### 3. 尋找合適的模型  
像 GPT-4o 和 GPT-4o mini 這些模型能理解和生成自然語言，且在 OpenAI 平台上可用，擁有不同的運算能力和速度，適用於不同的任務。


In [ ]:
# Select a general purpose chat model
model = "gpt-5-mini"


## 4. 提示設計  

「大型語言模型的魔力在於，透過在大量文字上訓練以最小化這種預測錯誤，模型最終學會了對這些預測有用的概念。例如，它們學會了類似」(1):

* 如何拼寫
* 語法如何運作
* 如何意譯
* 如何回答問題
* 如何進行對話
* 如何用多種語言寫作
* 如何編程
* 等等。

#### 如何控制大型語言模型  
「在所有輸入給大型語言模型的資料中，文本提示無疑是最具影響力的」(1)。

大型語言模型可以用幾種方式透過提示產生輸出：

指令：告訴模型你想要什麼
補完：誘導模型完成你想要的開始部分
示範：向模型展示你想要的，方式有：
提示中帶入幾個範例
調校訓練資料集中有數百或數千個範例」



#### 創建提示的三個基本指導原則：

<strong>說明並展示</strong>。透過指令、範例或兩者結合，讓你想要的內容清晰明確。如果你想讓模型按字母順序排列清單中的項目，或根據情感對段落進行分類，就讓它看到你想要的就是這樣。

<strong>提供優質資料</strong>。如果你想建立分類器或讓模型遵循某種模式，確保有足夠的範例。務必校對你的範例——模型通常聰明到能看穿基本的拼寫錯誤並給你回應，但它也可能認為這是故意為之，進而影響回應結果。

**檢查你的設定。** temperature 和 top_p 設定控制模型產生回應的決定性。如果你希望得到唯一正確答案，則應將這些設置調低。如果你想要更多樣化的回應，則可調高它們。人們在這些設定上最常犯的錯誤是假設它們是「聰明度」或「創造力」的控制。


來源：https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. 提交！


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### 重複相同的呼叫，結果有何不同？ 


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## 摘要文本  
#### 挑戰  
通過在文本段落末尾添加“tl;dr:”來摘要文本。注意模型如何在沒有額外指令的情況下理解並執行多項任務。你可以嘗試比 tl;dr 更具描述性的提示，以調整模型行為並自訂你所接收的摘要(3)。  

近期研究通過在大型語料庫上預訓練，隨後在具體任務上微調，顯示出在許多自然語言處理任務及基準上的顯著提升。雖然這種方法在架構上通常不針對特定任務，但仍需針對任務的數千或數萬個示例進行微調數據集。相比之下，人類通常只需少量示例或簡單指令即可完成新的語言任務——這是目前的自然語言處理系統仍普遍難以做到的。我們在此展示，擴大語言模型規模大幅提升了任務無關的少量示例表現，有時甚至能達到先前尖端微調方法的競爭力。  



簡而言之  


# 多個使用案例練習  
1. 總結文本  
2. 分類文本  
3. 生成新產品名稱


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## 分類文本  
#### 挑戰  
在推理時將項目分類到所提供的類別中。在以下示例中，我們在提示中同時提供類別和待分類的文本（*playground_reference）。 

客戶查詢：你好，我的筆記本電腦鍵盤有一顆鍵最近壞了，我需要更換：

分類類別：


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## 生成新產品名稱
#### 挑戰
從示例詞彙中創建產品名稱。這裡我們在提示中包含了有關我們將要為其生成名稱的產品的信息。我們還提供了一個類似的示例以展示我們希望收到的模式。我們同時將溫度值設定較高，以增加隨機性和更具創新性的回應。

產品描述：家用奶昔機
種子詞：快速、健康、緊湊。
產品名稱：HomeShaker、Fit Shaker、QuickShake、Shake Maker

產品描述：一雙能適合任何腳型的鞋子。
種子詞：適應性、合腳、全合腳。


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# 參考資料  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [用於文本分類的 GPT 模型微調最佳實踐](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# 如需更多幫助  
[OpenAI Commercialization Team](AzureOpenAITeam@microsoft.com) 


# 貢獻者
* Louis Li  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們力求準確，但請注意，自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議尋求專業人工翻譯。我們不對因使用本翻譯而引起的任何誤解或曲解承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
